In [29]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [30]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [31]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [32]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [33]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [34]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [35]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [36]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [37]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


In [38]:
num_states = len(states)         # 5
seq_length = len(query_sequence) # 26
MINUS_INF  = float('-inf')

# Shape:(num_states x seq_length)
viterbi_value_matrix = np.full((num_states, seq_length), MINUS_INF)
viterbi_trace_matrix = np.zeros((num_states, seq_length), dtype=int)

# Seed:start state is active at position 0 with log(0.25)
viterbi_value_matrix[states["s"], 0] = math.log(0.25)

print("Matrices initialised and the Shape:", viterbi_value_matrix.shape)

Matrices initialised and the Shape: (5, 26)


In [39]:
def calculate_prob_for_a_node(col, cur_state):
    nucleotide = query_sequence[col]
    nuc_idx= emission_nuc_codes[nucleotide]
    emit_prob = emission_probs[cur_state][nuc_idx]

    if emit_prob == 0:
        return MINUS_INF, 0   # ths state emission not possible

    log_emit_prob = math.log(emit_prob)
    best_score= MINUS_INF
    best_prev = 0

    for prev_state in range(num_states):
        trans_prob = state_transition_prob[prev_state][cur_state]
        if trans_prob == 0:
            continue
        prev_score = viterbi_value_matrix[prev_state, col - 1]
        if prev_score == MINUS_INF:
            continue
        candidate = prev_score + log_emit_prob + math.log(trans_prob)
        if candidate>best_score:
            best_score = candidate
            best_prev = prev_state

    return best_score, best_prev

In [40]:
for col in range(1, seq_length):
    for cur_state in range(num_states):
        score, best_prev = calculate_prob_for_a_node(col, cur_state)
        viterbi_value_matrix[cur_state, col]= score
        viterbi_trace_matrix[cur_state, col]= best_prev

print("Viterbi value matrix:")
print(np.array_str(viterbi_value_matrix, precision=6)) 

Viterbi value matrix:
[[ -1.386294       -inf       -inf       -inf       -inf       -inf
        -inf       -inf       -inf       -inf       -inf       -inf
        -inf       -inf       -inf       -inf       -inf       -inf
        -inf       -inf       -inf       -inf       -inf       -inf
        -inf       -inf]
 [      -inf  -2.772589  -4.264244  -5.755898  -7.247553  -8.739208
  -10.230863 -11.722518 -13.214173 -14.705828 -16.197483 -17.689137
  -19.180792 -20.672447 -22.164102 -23.655757 -25.147412 -26.639067
  -28.130722 -29.622377 -31.114031 -32.605686 -34.097341 -35.588996
  -37.080651 -38.572306]
 [      -inf       -inf       -inf       -inf -11.054216       -inf
  -11.093087       -inf -14.076396 -18.51249  -20.004145 -21.4958
  -20.043016       -inf -25.970765 -24.517981 -28.954074       -inf
  -28.992945       -inf -34.920694 -36.412349 -34.959565       -inf
        -inf -42.378968]
 [      -inf       -inf       -inf       -inf       -inf -11.970507
  -14.378452 -12.0093

In [41]:
def traceback_best_path():
    last_col_scores= viterbi_value_matrix[:, seq_length - 1]
    final_state = int(np.argmax(last_col_scores))
    path = [final_state]
    cur = final_state
    for col in range(seq_length - 1, 0, -1):
        cur = viterbi_trace_matrix[cur, col]
        path.append(cur)
    path.reverse()
    return "".join(id2state[i] for i in path)

optimal_path = traceback_best_path()
best_log_prob = np.max(viterbi_value_matrix[:, seq_length-1])

print(f"Query sequence: {query_sequence}")
print(f"Optimal path: {optimal_path}")
print(f"Log probability: {best_log_prob}")

Query sequence: CTTCATGTGAAAGCAGACGTAAGTCA
Optimal path: sEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.572305764904975
